In [1]:
# Install ipython-sql if not already installed
# !pip install ipython-sql

In [2]:
import sqlite3
import sql
import pandas as pd
import numpy as np

# Connect to a database (creates the database file if it doesn't exist)
cnn = sqlite3.connect('northwind1.db')

In [3]:
import sql

# Load the SQL extension
%load_ext sql

# Connect to a SQLite database
%sql sqlite:///northwind1.db

#display style of SQL query results in Jupyter Notebook
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [4]:
%%sql

SELECT name FROM sqlite_master WHERE type='table';

 * sqlite:///northwind1.db
Done.


name
sqlite_sequence
CustomerCustomerDemo
CustomerDemographics
EmployeeTerritories
Regions
Territories
Categories
Orders
Products
Shippers


In [4]:
from sqlalchemy import create_engine
import pandas as pd

# Connect to the SQLite database
engine = create_engine("sqlite:///northwind1.db")

# List of all tables
tables = [
     "EmployeeTerritories", "Regions", "Territories",
    "Categories", "Orders", "Products", "Shippers", "Suppliers", "Employees", "Customers"
]

# Loop through each table and display top 2 rows
for table in tables:
    print(f"\n--- {table} ---")
    df = pd.read_sql_query(f"SELECT * FROM {table} LIMIT 5;", engine)
    display(df)  # use display() in Jupyter to format nicely



--- EmployeeTerritories ---


,EmployeeID,TerritoryID
0,1,06897
1,1,19713
2,2,01581
3,2,01730
4,2,01833



--- Regions ---


,RegionID,RegionDescription
0,1,Eastern
1,2,Western
2,3,Northern
3,4,Southern



--- Territories ---


...

,SupplierID,SupplierName,ContactName,Address,City,PostalCode,Country,Phone
0,1,Exotic Liquids,Charlotte Cooper,49 Gilbert St.,London,EC1 4SD,UK,(171) 555-2222
1,2,New Orleans Cajun Delights,Shelley Burke,P.O. Box 78934,New Orleans,70117,USA,(100) 555-4822
2,3,Grandma Kelly's Homestead,Regina Murphy,707 Oxford Rd.,Ann Arbor,48104,USA,(313) 555-5735
3,4,Tokyo Traders,Yoshi Nagase,9-8 Sekimai\nMusashino-shi,Tokyo,100,Japan,(03) 3555-5011
4,5,Cooperativa de Quesos 'Las Cabras',Antonio del Valle Saavedra,Calle del Rosal 4,Oviedo,33007,Spain,(98) 598 76 54



--- Employees ---


,EmployeeID,LastName,FirstName,BirthDate,Notes
0,1,Davolio,Nancy,1968-12-08,Education includes a BA in psychology from Col...
1,2,Fuller,Andrew,1972-02-19,Andrew received his BTS commercial in 1974 and...
2,3,Leverling,Janet,1983-08-30,Janet has a BS degree in chemistry from Boston...
3,4,Peacock,Margaret,1957-09-19,Margaret holds a BA in English literature from...
4,5,Buchanan,Steven,1975-03-04,Steven Buchanan graduated from St. Andrews Uni...



--- Customers ---


,CustomerID,CustomerName,ContactName,Address,City,PostalCode,Country
0,1,Alfreds Futterkiste,Maria Anders,Obere Str. 57,Berlin,12209,Germany
1,2,Ana Trujillo Emparedados y helados,Ana Trujillo,Avda. de la Constitución 2222,México D.F.,05021,Mexico
2,3,Antonio Moreno Taquería,Antonio Moreno,Mataderos 2312,México D.F.,05023,Mexico
3,4,Around the Horn,Thomas Hardy,120 Hanover Sq.,London,WA1 1DP,UK
4,5,Berglunds snabbköp,Christina Berglund,Berguvsvägen 8,Luleå,S-958 22,Sweden


In [6]:
%%sql

SELECT * 
FROM Products;

 * sqlite:///northwind1.db
Done.


ProductID,ProductName,SupplierID,CategoryID,Unit,Price
1,Chai,1,1,10 boxes x 20 bags,18
2,Chang,1,1,24 - 12 oz bottles,19
3,Aniseed Syrup,1,2,12 - 550 ml bottles,10
4,Chef Anton's Cajun Seasoning,2,2,48 - 6 oz jars,22
5,Chef Anton's Gumbo Mix,2,2,36 boxes,21.35
6,Grandma's Boysenberry Spread,3,2,12 - 8 oz jars,25
7,Uncle Bob's Organic Dried Pears,3,7,12 - 1 lb pkgs.,30
8,Northwoods Cranberry Sauce,3,2,12 - 12 oz jars,40
9,Mishi Kobe Niku,4,6,18 - 500 g pkgs.,97
10,Ikura,4,8,12 - 200 ml jars,31


# 1️⃣ ROW_NUMBER()

In [7]:
%%sql

-- 👉 Assigns a unique row number to All products, ordered by price descending.


SELECT ProductID, ProductName, CategoryID, Price,
       ROW_NUMBER() OVER (ORDER BY Price DESC) AS RowNum
FROM Products;

 * sqlite:///northwind1.db
Done.


ProductID,ProductName,CategoryID,Price,RowNum
38,Côte de Blaye,1,263.5,1
29,Thüringer Rostbratwurst,6,123.79,2
9,Mishi Kobe Niku,6,97,3
20,Sir Rodney's Marmalade,3,81,4
18,Carnarvon Tigers,8,62.5,5
59,Raclette Courdavault,4,55,6
51,Manjimup Dried Apples,7,53,7
62,Tarte au sucre,3,49.3,8
43,Ipoh Coffee,1,46,9
28,Rössle Sauerkraut,7,45.6,10


In [5]:
%%sql

-- 👉 Gives each product a unique sequential number within its category

SELECT ProductID, ProductName, CategoryID, Price,
       ROW_NUMBER() OVER (PARTITION BY CategoryID ORDER BY Price DESC) AS RowNum
FROM Products;

 * sqlite:///northwind1.db
Done.


ProductID,ProductName,CategoryID,Price,RowNum
38,Côte de Blaye,1,263.5,1
43,Ipoh Coffee,1,46,2
2,Chang,1,19,3
1,Chai,1,18,4
35,Steeleye Stout,1,18,5
39,Chartreuse verte,1,18,6
76,Lakkalikööri,1,18,7
70,Outback Lager,1,15,8
34,Sasquatch Ale,1,14,9
67,Laughing Lumberjack Lager,1,14,10


# 2️⃣ RANK()

In [9]:
%%sql

-- 👉 Ranks All products across the entire table, same price = same rank, skips Next rank.


SELECT ProductID, ProductName, CategoryID, Price,
       RANK() OVER (ORDER BY Price DESC) AS RankVal
FROM Products;

 * sqlite:///northwind1.db
Done.


ProductID,ProductName,CategoryID,Price,RankVal
38,Côte de Blaye,1,263.5,1
29,Thüringer Rostbratwurst,6,123.79,2
9,Mishi Kobe Niku,6,97,3
20,Sir Rodney's Marmalade,3,81,4
18,Carnarvon Tigers,8,62.5,5
59,Raclette Courdavault,4,55,6
51,Manjimup Dried Apples,7,53,7
62,Tarte au sucre,3,49.3,8
43,Ipoh Coffee,1,46,9
28,Rössle Sauerkraut,7,45.6,10


In [10]:
%%sql

-- 👉 Products With the same price within a catagory get the same rank, but the Next rank IS skipped.


SELECT ProductID, ProductName, CategoryID, Price,
       RANK() OVER (PARTITION BY CategoryID ORDER BY Price DESC) AS RankVal
FROM Products;

 * sqlite:///northwind1.db
Done.


ProductID,ProductName,CategoryID,Price,RankVal
38,Côte de Blaye,1,263.5,1
43,Ipoh Coffee,1,46,2
2,Chang,1,19,3
1,Chai,1,18,4
35,Steeleye Stout,1,18,4
39,Chartreuse verte,1,18,4
76,Lakkalikööri,1,18,4
70,Outback Lager,1,15,8
34,Sasquatch Ale,1,14,9
67,Laughing Lumberjack Lager,1,14,9


#   3️⃣ DENSE_RANK()

In [11]:
%%sql

-- 👉 Same As RANK(), but doesn’t skip ranks


SELECT ProductID, ProductName, CategoryID, Price,
       DENSE_RANK() OVER (ORDER BY Price DESC) AS DenseRankVal
FROM Products;

 * sqlite:///northwind1.db
Done.


ProductID,ProductName,CategoryID,Price,DenseRankVal
38,Côte de Blaye,1,263.5,1
29,Thüringer Rostbratwurst,6,123.79,2
9,Mishi Kobe Niku,6,97,3
20,Sir Rodney's Marmalade,3,81,4
18,Carnarvon Tigers,8,62.5,5
59,Raclette Courdavault,4,55,6
51,Manjimup Dried Apples,7,53,7
62,Tarte au sucre,3,49.3,8
43,Ipoh Coffee,1,46,9
28,Rössle Sauerkraut,7,45.6,10


In [12]:
%%sql

-- 👉 Products With the same price get the same rank, And no rank Is skipped


SELECT ProductID, ProductName, CategoryID, Price,
       DENSE_RANK() OVER (PARTITION BY CategoryID ORDER BY Price DESC) AS DenseRankVal
FROM Products;

 * sqlite:///northwind1.db
Done.


ProductID,ProductName,CategoryID,Price,DenseRankVal
38,Côte de Blaye,1,263.5,1
43,Ipoh Coffee,1,46,2
2,Chang,1,19,3
1,Chai,1,18,4
35,Steeleye Stout,1,18,4
39,Chartreuse verte,1,18,4
76,Lakkalikööri,1,18,4
70,Outback Lager,1,15,5
34,Sasquatch Ale,1,14,6
67,Laughing Lumberjack Lager,1,14,6


# 4️⃣ NTILE(n)

In [8]:
%%sql

-- 👉 Divides All products into 4 groups based on price ordering.


SELECT ProductID, ProductName, CategoryID, Price,
       NTILE(4) OVER (ORDER BY Price DESC) AS PriceGroup
FROM Products;

 * sqlite:///northwind1.db
Done.


ProductID,ProductName,CategoryID,Price,PriceGroup
38,Côte de Blaye,1,263.5,1
29,Thüringer Rostbratwurst,6,123.79,1
9,Mishi Kobe Niku,6,97,1
20,Sir Rodney's Marmalade,3,81,1
18,Carnarvon Tigers,8,62.5,1
59,Raclette Courdavault,4,55,1
51,Manjimup Dried Apples,7,53,1
62,Tarte au sucre,3,49.3,1
43,Ipoh Coffee,1,46,1
28,Rössle Sauerkraut,7,45.6,1


In [9]:
%%sql

-- 👉 Splits products into 3 price-based groups within each category


SELECT ProductID, ProductName, CategoryID, Price,
       NTILE(3) OVER (PARTITION BY CategoryID ORDER BY Price DESC) AS PriceGroup
FROM Products;

 * sqlite:///northwind1.db
Done.


ProductID,ProductName,CategoryID,Price,PriceGroup
38,Côte de Blaye,1,263.5,1
43,Ipoh Coffee,1,46,1
2,Chang,1,19,1
1,Chai,1,18,1
35,Steeleye Stout,1,18,2
39,Chartreuse verte,1,18,2
76,Lakkalikööri,1,18,2
70,Outback Lager,1,15,2
34,Sasquatch Ale,1,14,3
67,Laughing Lumberjack Lager,1,14,3


In [10]:
%%sql

-- 👉 Divides All products into 10 groups based on price ordering.


SELECT ProductID, ProductName, CategoryID, Price,

    ROW_NUMBER() OVER (ORDER BY Price DESC) AS RowNum,
    
    RANK() OVER (ORDER BY Price DESC) AS RANK,
    
    DENSE_RANK() OVER (ORDER BY Price DESC) AS DENSE_RANK,
    NTILE(10) OVER (ORDER BY Price DESC) AS PriceGroup
FROM Products;

 * sqlite:///northwind1.db
Done.


ProductID,ProductName,CategoryID,Price,RowNum,RANK,DENSE_RANK,PriceGroup
38,Côte de Blaye,1,263.5,1,1,1,1
29,Thüringer Rostbratwurst,6,123.79,2,2,2,1
9,Mishi Kobe Niku,6,97,3,3,3,1
20,Sir Rodney's Marmalade,3,81,4,4,4,1
18,Carnarvon Tigers,8,62.5,5,5,5,1
59,Raclette Courdavault,4,55,6,6,6,1
51,Manjimup Dried Apples,7,53,7,7,7,1
62,Tarte au sucre,3,49.3,8,8,8,1
43,Ipoh Coffee,1,46,9,9,9,2
28,Rössle Sauerkraut,7,45.6,10,10,10,2


# The notebook ends here....!

# 5️⃣ 2nd cheapest product overall

In [ ]:
%%sql

-- 👉 Adds a column that always shows the second cheapest product overall.


SELECT 
    CategoryID,
    ProductID,
    ProductName,
    Price,
    NTH_VALUE(ProductName, 2) OVER (
        Partition by CategoryID
        ORDER BY Price 
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS second_cheapest_product
FROM Products;


# 6️⃣ MIN() and MAX() — Price Range per Category

In [ ]:
%%sql

-- 👉 Finds the second most expensive product within each CategoryID.

SELECT 
    CategoryID,
    ProductName,
    Price,
    NTH_VALUE(ProductName, 2) OVER (
        PARTITION BY CategoryID
        ORDER BY Price DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS second_costliest_in_category
FROM Products;

# 7️⃣ 🏆 Advanced Example – Combining all three

#### Let’s compare the cheapest, most expensive, and 2nd cheapest product per supplier.

In [ ]:
%%sql

-- 👉 This gives a full summary per supplier using FIRST_VALUE, LAST_VALUE, and NTH_VALUE together.


SELECT 
    SupplierID,
    ProductName,
    Price,
    
    FIRST_VALUE(ProductName) OVER (
        PARTITION BY SupplierID ORDER BY Price
    ) AS cheapest_product,
    
    
    LAST_VALUE(ProductName) OVER (
        PARTITION BY SupplierID ORDER BY Price
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS costliest_product,
    
    
    NTH_VALUE(ProductName, 2) OVER (
        PARTITION BY SupplierID ORDER BY Price
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS second_cheapest_product
    
FROM Products;


# Notebook Ends Here.

# 8️⃣ Combining Multiple Window Aggregates

In [ ]:
%%sql
SELECT 
    ProductID,
    ProductName,
    CategoryID,
    Price,
    SUM(Price) OVER () AS total_price_all_products,
    COUNT(*) OVER (PARTITION BY CategoryID) AS product_count,
    SUM(Price) OVER (PARTITION BY CategoryID) AS total_price,
    AVG(Price) OVER (PARTITION BY CategoryID) AS avg_price,
    MIN(Price) OVER (PARTITION BY CategoryID) AS min_price,
    MAX(Price) OVER (PARTITION BY CategoryID) AS max_price
FROM Products;

In [ ]:
%%sql
SELECT 
    CategoryID,
    COUNT(*) AS product_count,
    SUM(Price) AS total_price,
    AVG(Price) AS avg_price,
    MIN(Price) AS min_price,
    MAX(Price) AS max_price
FROM Products
GROUP BY CategoryID;

# 9️⃣ COUNT() with Running Count (ROW-Like Behavior)

In [ ]:
%%sql
SELECT 
    ProductID,
    ProductName,
    CategoryID,
    Price,
    COUNT(*) OVER (
        PARTITION BY CategoryID
        ORDER BY ProductID
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_product_count
FROM Products;

GROUP BY Equivalent

❌ Like cumulative sums, running counts cannot be done with plain GROUP BY.
We can use correlated subquery:

In [ ]:
%%sql
SELECT 
    p1.ProductID,
    p1.ProductName,
    p1.CategoryID,
    p1.Price,
    (
        SELECT COUNT(*)
        FROM Products p2
        WHERE p2.CategoryID = p1.CategoryID
          AND p2.ProductID <= p1.ProductID
    ) AS running_product_count
FROM Products p1
ORDER BY p1.CategoryID, p1.ProductID;

In [ ]:
%%sql

-- 👉 Gives each product a unique sequential number within its category

SELECT ProductID, ProductName, CategoryID, Price,
       ROW_NUMBER() OVER (ORDER BY Price DESC) AS RowNum
FROM Products;

# Count the number of customers per country

In [ ]:
%%sql

SELECT 

Country, 

COUNT(*) AS Total_Customers

FROM Customers

GROUP BY Country;

In [ ]:
%%sql

SELECT 

COUNT(*) AS Total_Customers

FROM Customers;

In [ ]:
%%sql

SELECT 
Country, 
COUNT(*) AS Total_Customers
FROM Customers
GROUP BY Country
ORDER BY Total_Customers
desc;

# Get the average price of the product in each category

In [ ]:
%%sql

SELECT *
FROM Products
Limit 5;

In [ ]:
%%sql

SELECT AVG(Price) AS AvgPrice 
FROM Products;

In [ ]:
%%sql

SELECT AVG(Price) AS AvgPrice 
FROM Products
group by CategoryID;

In [ ]:
%%sql

SELECT CategoryID, AVG(Price) AS AvgPrice 
FROM Products
group by CategoryID;

In [ ]:
%%sql

select * FROM categories;

In [ ]:
%%sql

SELECT CategoryID, AVG(Price) AS AvgPrice 
FROM Products
group by CategoryID
order by AVG(Price);

In [ ]:
%%sql

SELECT CategoryID, AVG(Price) AS AvgPrice 
FROM Products
group by CategoryID
order by AVG(Price);

## repeate same for sum of price within each category

# Get Minimum and Maxmium Price per Category

In [ ]:
%%sql

SELECT CategoryID, 
MAX(Price) AS Max_Price, 
MIN(Price) AS Min_Price

FROM Products
GROUP BY CategoryID;

In [ ]:
%%sql

select * FROM categories;

In [ ]:
%%sql

SELECT 
    c.CategoryName,
    p.CategoryID, 
    MAX(p.Price) AS Max_Price, 
    MIN(p.Price) AS Min_Price
FROM Products p
JOIN Categories c 
    ON p.CategoryID = c.CategoryID
GROUP BY 
    c.CategoryName, 
    p.CategoryID;

In [ ]:
%%sql

SELECT country,
city,
count(*) AS Total_customers_per_country_per_city
From Customers

group by country, city;

In [ ]:
%%sql

SELECT country, city, count(*) AS total_cust

From Customers

group by country, city

having count(*) > 2

order by count(*) asc;

# 🚨 Common Mistakes
## ❌ Using columns in SELECT that are not in GROUP BY or aggregate functions.

## ❌ Using WHERE instead of HAVING for aggregated values.

# ✅ Best Practices

### Always use aggregate functions with columns not in the GROUP BY clause.

### Use HAVING to filter aggregated results.

### Use aliases to improve readability of output.

In [ ]:
%%sql

SELECT * 
FROM Suppliers
Limit 5;

# Get all suppliers and customers from London. (Using Multiple CTEs)

In [ ]:
%%sql

WITH LondonCustomers AS (
    SELECT CustomerID, ContactName, City FROM Customers WHERE City = 'London'
),
LondonSuppliers AS (
    SELECT SupplierID, ContactName, City FROM  WHERE City =Suppliers 'London'
)
SELECT * FROM LondonCustomers
UNION ALL
SELECT * FROM LondonSuppliers;

In [ ]:
%%sql

WITH LondonCustomers AS (
    SELECT 
        CustomerID AS ID, 
        ContactName, 
        City, 
        'Customer' AS Type
    FROM Customers 
    WHERE City = 'London'
),
LondonSuppliers AS (
    SELECT 
        SupplierID AS ID, 
        ContactName, 
        City, 
        'Supplier' AS Type
    FROM Suppliers 
    WHERE City = 'London'
)
SELECT * FROM LondonCustomers
UNION ALL
SELECT * FROM LondonSuppliers;

In [ ]:
%%sql

SELECT * FROM Products
Limit 10;

In [ ]:
%%sql

WITH AvgPriceCTE AS (
    SELECT AVG(Price) AS AvgPrice FROM Products
)
SELECT *
FROM Products
WHERE Price > (SELECT AvgPrice FROM AvgPriceCTE);

# 📌 This gives you the number of products in each category.

In [ ]:
%%sql

WITH CategoryCounts AS (
    SELECT CategoryID, COUNT(*) AS TotalProducts
    FROM Products
    GROUP BY CategoryID
)
SELECT *
FROM CategoryCounts;

# CTE to Identify Duplicate Prices
📌 Finds all products that share the same price as others (duplicates).

In [ ]:
%%sql

WITH PriceCounts AS (
    SELECT Price, COUNT(*) AS Occurrences
    FROM Products
    GROUP BY Price
    HAVING COUNT(*) > 1
)
SELECT *
FROM Products
WHERE Price IN (SELECT Price FROM PriceCounts) order by price;

# CTE to Get the Cheapest Product Per Category

In [ ]:
%%sql

SELECT CategoryID, MIN(Price) AS MinPrice
    FROM Products
    GROUP BY CategoryID;

In [ ]:
%%sql

WITH MinPrices AS (
    SELECT CategoryID, MIN(Price) AS MinPrice
    FROM Products
    GROUP BY CategoryID
)
SELECT *
FROM Products
WHERE (CategoryID, Price) IN (
    SELECT CategoryID, MinPrice FROM MinPrices
) order by CategoryID;

# Get the employees who handle more than 2 territories.

In [ ]:
%%sql

WITH TerritoryCount AS (
    SELECT EmployeeID, COUNT(*) AS TotalTerritories
    FROM EmployeeTerritories
    GROUP BY EmployeeID
)
SELECT E.FirstName, E.LastName, T.TotalTerritories
FROM Employees E
JOIN TerritoryCount T ON E.EmployeeID = T.EmployeeID
WHERE T.TotalTerritories > 2;

# 📌 Returns the most expensive product in each category.

In [ ]:
%%sql

WITH RankedCategory AS (
    SELECT ProductID, ProductName, CategoryID, Price,
           RANK() OVER (PARTITION BY CategoryID ORDER BY Price DESC) AS RankInCategory
    FROM Products
)
SELECT *
FROM RankedCategory
WHERE RankInCategory = 1;

# 📌 Filters category 3 products, then returns the top 3 most expensive among them.

In [ ]:
%%sql

WITH Filtered AS (
    SELECT * FROM Products WHERE CategoryID = 3
),
Ranked AS (
    SELECT ProductID, ProductName, Price,
           DENSE_RANK() OVER (ORDER BY Price DESC) AS PriceRank
    FROM Filtered
)
SELECT * FROM Ranked WHERE PriceRank <= 3;

In [ ]:
%%sql

SELECT * FROM Products
ORDER BY ProductName;

# 📌 Categorizes products into 'Low', 'Medium', and 'High' price brackets.

In [ ]:
%%sql

WITH PriceBuckets AS (
    SELECT ProductID, ProductName, Price,
           CASE 
               WHEN Price < 10 THEN 'Low'
               WHEN Price BETWEEN 10 AND 30 THEN 'Medium'
               ELSE 'High'
           END AS PriceCategory
    FROM Products
)
SELECT *
FROM PriceBuckets
WHERE PriceCategory = 'Low';

# Recursive CTEs

In [ ]:
%%sql

WITH RECURSIVE FactorialCTE(n, fact) AS (
    SELECT 1, 1
    UNION ALL
    SELECT n + 1, (n + 1) * fact
    FROM FactorialCTE
    WHERE n < 5
)
SELECT * FROM FactorialCTE;

# Return all customers that starts with "E" and are at least 3 characters in length

In [ ]:
%%sql

SELECT * FROM Customers
WHERE CustomerName LIKE 'E__%';

# END..... Practice Below...

In [ ]:
%%sql

SELECT * 
FROM Customers
WHERE country = 'Brazil';

In [ ]:
%%sql

SELECT * 
FROM Customers
WHERE CustomerID=3;

In [ ]:
%%sql

SELECT * FROM Products;

In [ ]:
%%sql

SELECT *
FROM Products
WHERE Price > 40;

In [ ]:
%%sql

SELECT * 
FROM Products
WHERE Price < 40;

In [ ]:
%%sql

SELECT * 
FROM Products
WHERE Price >= 30;

In [ ]:
%%sql

SELECT * 
FROM Products
WHERE Price <= 30;

In [ ]:
%%sql

SELECT * FROM Products
WHERE Price <> 30;

In [ ]:
%%sql

SELECT * 
FROM Products
WHERE Price BETWEEN 30 AND 40;

In [ ]:
%%sql

SELECT * 
FROM Customers
WHERE City IN ('Paris','London');

In [ ]:
%%sql



In [ ]:
%%sql



In [ ]:
%%sql



In [ ]:
%%sql



In [ ]:
%%sql

